Importação de bibliotecas

## Etapa 1 - Parametros do experimento da pandemia

Execute a celula abaixo para conferir as datas e a pasta de dados, sem baixar nada. As datas ficam em `parametros_experimento.json`, na pasta do projeto.

Sua parte utiliza 2017-2019 para construir a MST e selecionar os ativos. O periodo 2020-2022 e uma referencia para entrega ao integrante responsavel pela avaliacao financeira. O planejamento esta em `PLANO_ALGORITMOS.md`.

Esta etapa ainda nao recorta Pearson nem seleciona ativos elegiveis.


In [24]:
from pathlib import Path
import json

parametros = json.loads(Path('parametros_experimento.json').read_text(encoding='utf-8'))
inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
inicio_avaliacao = parametros['inicio_avaliacao']
fim_avaliacao = parametros['fim_avaliacao']

print('Formacao da MST:', inicio_formacao, 'ate', fim_formacao)
print('Avaliacao financeira (outro integrante):', inicio_avaliacao, 'ate', fim_avaliacao)
pasta_coleta = Path(parametros['pasta_coleta'])
print('Pasta da coleta:', pasta_coleta.resolve())
if (pasta_coleta / 'acoes.csv').exists() and (pasta_coleta / 'metadados.json').exists():
    print('Coleta local encontrada; a celula de dados verificara sua integridade.')
else:
    print('Coleta local ainda nao esta completa. Esta celula nao realiza downloads.')


Formacao da MST: 2017-01-01 ate 2019-12-31
Avaliacao financeira (outro integrante): 2020-01-01 ate 2022-12-31
Pasta da coleta: C:\Users\cida3\OneDrive\Documentos\TrabalhoDiscreta\dados\pandemia
Coleta local ainda nao esta completa. Esta celula nao realiza downloads.


### Continuacao: importacoes e dados
A celula acima funciona sozinha. Executar as celulas seguintes pode instalar bibliotecas e consultar servicos externos.

In [25]:
import numpy as np
import pandas as pd

%run ./Tabelas_Acoes.ipynb

Formacao da MST: 2017-01-01 ate 2019-12-31
Avaliacao financeira (outro integrante): 2020-01-01 ate 2022-12-31
Pasta da coleta: C:\Users\cida3\OneDrive\Documentos\TrabalhoDiscreta\dados\pandemia
Coleta local ainda nao esta completa. Esta celula nao realiza downloads.


$CASN3.SA: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
$CASN4.SA: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
$AZUL3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$AUAU3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

4 Failed downloads:
['CASN3.SA', 'CASN4.SA']: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
['AZUL3.SA', 'AUAU3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$CYRE4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$HETA3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$EGGY3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

3 Failed downloads:
['CYRE4.SA', 'HETA3.SA', 'EGGY3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$ODER4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$PASS3.SA: Data doesn't ex

Nova coleta salva: dados\pandemia\acoes.csv
Data da coleta (UTC): 2026-11-16T22:11:41.016388+00:00


c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


## Etapa 2 - Elegibilidade e retornos da formacao (2017-2019)

Execute as duas celulas abaixo depois de carregar `df` e `acoes`, com NumPy e Pandas importados. Se os dados ja estao no kernel, nao e necessario repetir o `%run`.

A cobertura minima esta em `parametros_experimento.json`. As datas da base sao uma referencia provisoria de sessoes, nao um calendario oficial da B3. A funcao permite receber um calendario pelo argumento `sessoes`.

`Elegivel` significa que a acao atende aos criterios observaveis nos dados; nao certifica a data do IPO. Falta de evidencia no inicio, ausencia de preco na ultima sessao ou precos invalidos geram pendencia, salvo se ja houver motivo numerico de exclusao. Mudancas de ticker e o universo historico precisam de verificacao com o integrante responsavel pelos dados.

A base `df` e preservada. Sao criados `tabela_elegibilidade`, `retornos_formacao`, `acoes_elegiveis` e `retornos_formacao_elegiveis`. A tabela inclui tambem tickers solicitados sem fechamento disponivel. Nao sao usados precos posteriores a 2019. A tabela e os retornos ficam em memoria; a exportacao pertence a etapa 7.

**Confira a tabela antes de prosseguir. As etapas seguintes usam apenas os retornos elegiveis desta formacao.**


In [26]:
def preparar_formacao(precos, inicio, fim, cobertura_minima=0.9,
                      tickers_solicitados=None, sessoes=None):
    if not 0 < cobertura_minima <= 1:
        raise ValueError('A cobertura minima deve estar entre 0 e 1.')
    if not isinstance(precos.index, pd.DatetimeIndex) or not precos.index.is_unique:
        raise ValueError('Os precos precisam de datas unicas em um DatetimeIndex.')
    if not precos.columns.is_unique:
        raise ValueError('Os tickers nao podem estar duplicados.')
    inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
    if inicio > fim:
        raise ValueError('O inicio da formacao deve ser anterior ao fim.')

    # Descarta o futuro antes de qualquer decisao de elegibilidade.
    precos = precos.sort_index().loc[:fim].copy()
    tickers = set(precos.columns)
    if tickers_solicitados is not None:
        tickers.update(tickers_solicitados)
    precos = precos.reindex(columns=sorted(tickers))

    referencia = 'datas da base (provisorio)' if sessoes is None else 'calendario fornecido'
    if sessoes is None:
        sessoes = precos.index
    sessoes = pd.DatetimeIndex(sessoes).unique().sort_values()
    sessoes = sessoes[(sessoes >= inicio) & (sessoes <= fim)]
    if len(sessoes) == 0:
        raise ValueError('Nao ha datas de referencia no periodo de formacao.')

    # Insere sessoes ausentes antes da diferenca, sem preencher precos.
    precos = precos.reindex(precos.index.union(sessoes).sort_values())
    invalidos = precos.notna() & (~np.isfinite(precos) | (precos <= 0))
    precos_validos = precos.where(np.isfinite(precos) & (precos > 0))
    precos_janela = precos_validos.reindex(sessoes)
    # O primeiro retorno pode usar o fechamento anterior a 2017.
    retornos = np.log(precos_validos).diff().reindex(sessoes)

    tabela = pd.DataFrame({
        'Primeiro_preco_na_formacao': precos_janela.apply(lambda serie: serie.first_valid_index()),
        'Ultimo_preco_na_formacao': precos_janela.apply(lambda serie: serie.last_valid_index()),
        'Historico_no_inicio': precos_validos.loc[:sessoes[0]].notna().any(),
        'Preco_na_ultima_sessao': precos_janela.iloc[-1].notna(),
        'Retornos_validos': retornos.count(),
        'Sessoes_referencia': len(sessoes),
        'Cobertura': retornos.count() / len(sessoes),
        'Variancia': retornos.var(),
        'Precos_invalidos': invalidos.reindex(sessoes).sum()
    })
    tabela.index.name = 'Ticker'
    tabela['Situacao'] = 'Elegivel'
    tabela['Motivo'] = 'Atende aos criterios observaveis na base; listagem oficial nao verificada.'
    for ticker, linha in tabela.iterrows():
        exclusoes, pendencias = [], []
        if linha['Retornos_validos'] < 2:
            exclusoes.append('Menos de dois retornos validos na formacao')
        elif not np.isfinite(linha['Variancia']) or linha['Variancia'] <= 0:
            exclusoes.append('Variancia nula ou indefinida')
        if linha['Cobertura'] < cobertura_minima:
            exclusoes.append('Cobertura abaixo do minimo')
        if not linha['Historico_no_inicio']:
            pendencias.append('Sem evidencia de negociacao no inicio; investigar historico ou ticker')
        if not linha['Preco_na_ultima_sessao']:
            pendencias.append('Sem preco na ultima sessao de referencia; investigar')
        if linha['Precos_invalidos'] > 0:
            pendencias.append('Precos nao positivos ou infinitos na formacao; investigar')
        if exclusoes or pendencias:
            tabela.loc[ticker, 'Situacao'] = 'Excluida' if exclusoes else 'Pendente'
            tabela.loc[ticker, 'Motivo'] = '; '.join(exclusoes + pendencias)
    tabela.attrs['referencia_calendario'] = referencia
    return tabela, retornos


In [ ]:
# Pode ser executada com df ja carregado, sem repetir o %run/download.
from pathlib import Path
import json

parametros = json.loads(Path('parametros_experimento.json').read_text(encoding='utf-8'))
inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
tabela_elegibilidade, retornos_formacao = preparar_formacao(
    df['Close'], inicio_formacao, fim_formacao,
    cobertura_minima=parametros['cobertura_minima_formacao'],
    tickers_solicitados=acoes
)
acoes_elegiveis = tabela_elegibilidade.index[
    tabela_elegibilidade['Situacao'] == 'Elegivel'
].tolist()
retornos_formacao_elegiveis = retornos_formacao[acoes_elegiveis].copy()

print('Formacao:', inicio_formacao, 'ate', fim_formacao)
print('Referencia das sessoes:', tabela_elegibilidade.attrs['referencia_calendario'])
print(tabela_elegibilidade['Situacao'].value_counts())
display(tabela_elegibilidade)
print('Entrada da etapa 3:', retornos_formacao_elegiveis.shape, '(datas, ativos elegiveis)')


## Etapa 3 - Pearson e distancias de Mantegna da formacao

As matrizes abaixo usam somente `retornos_formacao_elegiveis` de 2017-2019. O minimo de observacoes conjuntas por par e definido no arquivo de parametros. As funcoes de Kruskal e metricas existentes serao reaplicadas nas etapas 4 e 5; Prim permanece sem chamada ativa.


In [ ]:
# Correlacoes dos retornos elegiveis da formacao, sem dados da avaliacao.
min_observacoes_conjuntas = int(np.ceil(
    len(retornos_formacao_elegiveis)
    * parametros['proporcao_minima_observacoes_conjuntas']
))
matriz_correlacoes_formacao = retornos_formacao_elegiveis.corr(
    method='pearson', min_periods=min_observacoes_conjuntas
)
matriz_correlacoes_formacao


In [ ]:
def matriz_mantegna(matriz_pearson):
     return np.sqrt(2*(1-matriz_pearson))

matriz_mantegna_formacao = matriz_mantegna(matriz_correlacoes_formacao)
matriz_mantegna_formacao


In [ ]:
# Ramo exploratorio de IPCA: nao utilizado na MST da formacao.
dataframe_IPCA = inflacao_mensal('01/01/2008', '31/12/2025', variacao_logaritma)
matriz_correlacao_IPCA = dataframe_IPCA.corr(method='pearson')
matriz_mantegna_IPCA = matriz_mantegna(matriz_correlacao_IPCA)
matriz_mantegna_IPCA


In [ ]:
def validar_matriz(df):
    # NaN representa uma distancia desconhecida, nunca uma aresta de peso zero.
    if not isinstance(df, pd.DataFrame):
        raise ValueError('A matriz deve ser um DataFrame.')
    if df.empty or df.shape[0] != df.shape[1]:
        raise ValueError('A matriz deve ser quadrada e nao vazia.')
    if not df.index.equals(df.columns) or not df.index.is_unique:
        raise ValueError('Linhas e colunas devem ter os mesmos ativos, sem repeticao e na mesma ordem.')
    valores = df.to_numpy()
    if valores.dtype.kind not in 'fiu':
        raise ValueError('As distancias devem ser numeros reais.')
    if np.isinf(valores).any():
        raise ValueError('A matriz contem distancias infinitas.')
    if not np.allclose(valores, valores.T, rtol=0, atol=1e-10, equal_nan=True):
        raise ValueError('A matriz deve ser simetrica, inclusive nas posicoes ausentes.')
    diagonal = np.diag(valores)
    if not np.all(np.isnan(diagonal) | np.isclose(diagonal, 0, rtol=0, atol=1e-10)):
        raise ValueError('As distancias conhecidas na diagonal devem ser zero.')
    conhecidos = valores[np.isfinite(valores)]
    if ((conhecidos < 0) | (conhecidos > 2)).any():
        raise ValueError('As distancias de Mantegna devem estar entre 0 e 2.')


def criar_arestas(df):
    validar_matriz(df)
    n=len(df.columns)
    linha = 0
    arestas = []
    for j in range(1,n):
        for i in range(linha+1):
            peso = df.iloc[i,j]
            if np.isfinite(peso):
                arestas.append((peso,i,j ))
        linha+=1
    
    return sorted(arestas)
arestas = criar_arestas(matriz_mantegna_formacao)
print(arestas)


In [32]:
def find(lista, vertice):
    if lista[vertice] == vertice:
        return vertice
    while lista[vertice] != vertice:
        vertice = lista[vertice]
    return vertice

def union(lista, vertice1, vertice2):
    pai1 = find(lista, vertice1)
    pai2 = find(lista, vertice2)
    if pai1 != pai2:
         lista[pai2] = pai1

def kruskal(df):
    arestas = criar_arestas(df)
    n = len(df.columns)
    pais = list(range(n))
    mst = []
    for peso, vertice1, vertice2 in arestas:
        if find(pais, vertice1) != find(pais, vertice2):
            union(pais, vertice1, vertice2)
            mst.append((peso, vertice1, vertice2))
        if len(mst) == n-1:
            break
    if len(mst) != n-1:
        raise ValueError('Grafo desconexo: as distancias conhecidas nao conectam todos os ativos.')
    return mst


In [33]:
def prim(df):
    validar_matriz(df)
    n = len(df.columns)
    vertice_inicial = 0
    visitados = set()
    visitados.add(vertice_inicial)
    mst = []
    while len(visitados) < n:
        menor_peso = (np.inf, None, None)
        for u in visitados:
            for v in range(n):
                if v not in visitados:
                    peso = df.iloc[u,v]
                    if np.isfinite(peso) and peso < menor_peso[0]:
                        menor_peso = (peso,u,v)
        if menor_peso[2] is None:
            raise ValueError('Grafo desconexo: as distancias conhecidas nao conectam todos os ativos.')
        visitados.add(menor_peso[2])
        mst.append(menor_peso)
    
    return mst


In [ ]:
# Etapa 4 - MST da formacao; a chamada de Prim permanece comentada.
#mst_prim = prim(matriz_mantegna_formacao)
mst_kruskal = kruskal(matriz_mantegna_formacao)

#print("Arestas Prim:", len(mst_prim))
print("Arestas Kruskal:", len(mst_kruskal))

#peso_prim = sum(peso for peso, u, v in mst_prim)
peso_kruskal = sum(peso for peso, u, v in mst_kruskal)

#print("Peso total Prim:", peso_prim)
print("Peso total Kruskal:", peso_kruskal)

In [36]:
def criar_dicionario_adjacencia(mst):
    adjacencia = {}
    for peso, v1, v2 in mst:
        adjacencia.setdefault(v1,[]).append((peso,v2))
        adjacencia.setdefault(v2,[]).append((peso,v1))
    return adjacencia

def calcular_graus(adjacencia):
    graus = {}
    for vertice in adjacencia:
        graus[vertice] = len(adjacencia[vertice])
    return graus


def calcular_excentricidade(adjacencia):
    excentricidades = {}

    for origem in adjacencia:
        distancias = {origem: 0}
        pilha = [origem]

        while pilha:
            vertice = pilha.pop()

            for peso, vizinho in adjacencia[vertice]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[vertice] + peso
                    pilha.append(vizinho)

        if len(distancias) != len(adjacencia):
            raise ValueError("A árvore precisa ser conexa.")

        excentricidades[origem] = max(distancias.values())

    return excentricidades


def distancia_entre_vertices(adjacencia, origem, destino):
    if origem not in adjacencia or destino not in adjacencia:
        raise ValueError('Os dois vertices devem pertencer a arvore.')
    distancias = {origem: 0}
    pilha = [origem]
    while pilha:
        vertice = pilha.pop()
        if vertice == destino:
            return distancias[vertice]
        for peso, vizinho in adjacencia[vertice]:
            if vizinho not in distancias:
                distancias[vizinho] = distancias[vertice] + peso
                pilha.append(vizinho)
    raise ValueError('Nao existe caminho entre os vertices informados.')


def calcular_diametro(adjacencia):
    if not adjacencia:
        raise ValueError('A arvore nao pode ser vazia.')
    excentricidades = calcular_excentricidade(adjacencia)
    return max(excentricidades.values())


def calcular_centro(adjacencia):
    if not adjacencia:
        raise ValueError('A arvore nao pode ser vazia.')
    excentricidades = calcular_excentricidade(adjacencia)
    menor_excentricidade = min(excentricidades.values())
    centro = []
    for vertice, excentricidade in excentricidades.items():
        if abs(excentricidade - menor_excentricidade) <= 1e-10:
            centro.append(vertice)
    return centro

In [ ]:
# Etapa 5 - Metricas estruturais da MST da formacao.
adjacencia = criar_dicionario_adjacencia(mst_kruskal)
graus = calcular_graus(adjacencia)
excentricidades = calcular_excentricidade(adjacencia)
diametro = calcular_diametro(adjacencia)
centro = calcular_centro(adjacencia)
vertices = sorted(adjacencia)

metricas = pd.DataFrame({
    'Vertice': vertices,
    'Ticker': [matriz_mantegna_formacao.columns[v] for v in vertices],
    'Grau': [graus[v] for v in vertices],
    'Excentricidade': [excentricidades[v] for v in vertices],
    'Folha': [graus[v] == 1 for v in vertices],
    'Centro': [v in centro for v in vertices]
})

display(metricas.sort_values(['Excentricidade', 'Ticker']))
print('Diametro:', diametro)
print('Centro:', [matriz_mantegna_formacao.columns[v] for v in centro])
